# Welcome to plantGREP Colab

plantGREP Colab is a Google Colab notebook designed to run plantGREP (plant Gene Regulatory Element Predictor), a deep learning model for predicting enhancer strength, developed in [Jores et al., 2026, unpublished](<DOI>).

plantGREP Colab offers an easy-to-use, code-free interface to:
- predict enhancer strength (prediction)
- identify the underlying functional sequence motifs (DeepLIFT)
- improve enhancer activity (evolution)

## Usage

First, you need to initialize the notebook by running the `Run plantGREP Colab` cell below twice. You can run the cell by clicking the image.png button under the cell title or by clicking on `Run all` in the title bar. In the first run, new modules will be installed (this will take about 2-3 minutes) and the session will crash (on purpose). After this, re-run the cell and after 30-60 seconds you should see the main user interface underneath the cell.

To use plantGREP Colab:
1. copy your sequence(s) in [FASTA format](https://en.wikipedia.org/wiki/FASTA_format) into the `Sequences:` text box
1. select a mode: `prediction`, `DeepLIFT`, or `evolution`
1. choose additional parameters (e.g., target condition; parameters depend on the selected mode)
1. run plantGREP by clicking the `predict strength`/`run DeepLIFT`/`evolve sequence` button

After completing the task, the results are shown below as an interactive table and plot. Use the `download results` button to download the results table (in tsv format) or the `download plot` button to download the current plot (in pdf format).

Please cite [Jores et al., 2026, unpublished](<DOI>) when using this notebook. Thank you!

In [ ]:
# @title Run plantGREP Colab
%%capture --no-display
%xmode Minimal
# "%xmode Minimal" suppresses traceback for errors

try:
  # import modules
  import os
  import warnings
  import torch
  import pytorch_lightning
  import logomaker
  import pandas as pd
  import seqpro as sp
  import ipywidgets as widgets
  from urllib.request import urlretrieve
  from google.colab import data_table, files
  from captum.attr import DeepLift
  from google.colab import files
  from IPython.display import display, HTML

  # download files files from GitHub repository
  github_name = 'plantGREP'
  github_url = f'https://raw.githubusercontent.com/tobjores/{github_name}/refs/heads/main/code/plantGREPcli/'
  ext_files = ['plantGREPcli.py', 'plantGREP_package.pt', 'baselines.npy']
  for filename in ext_files:
    urlretrieve(github_url + filename, filename)

  # import downloaded module
  import plantGREPcli

  # show only SequenceWarnings
  warnings.simplefilter('ignore')
  warnings.simplefilter('always', category = plantGREPcli.SequenceWarning)

  # display pandas dataframes as interactive tables
  data_table.enable_dataframe_formatter()

  # load plantGREP model
  model = plantGREPcli.load_model()
  if torch.cuda.is_available():
    device = torch.device('cuda')
    model.to(device)
  else:
    device = torch.device('cpu')

  # set up dictionary for results
  results = {
    'prediction' : pd.DataFrame(),
    'DeepLIFT' : pd.DataFrame(),
    'evolution' : pd.DataFrame(),
  }
  plots = {
    'prediction' : {},
    'DeepLIFT' : {},
    'evolution' : {}
  }

  # define widgets
  seq_input = widgets.Textarea(
    description = 'Sequences:',
    placeholder = 'paste fasta sequences here',
    layout = widgets.Layout(width = '100%', height = '10em', margin = '0px 0px 20px 0px')
  )

  tb_mode = widgets.ToggleButtons(
    options = ['prediction', 'DeepLIFT', 'evolution'],
    description = 'Mode:',
    tooltips = ['predict enhancer strength', 'perform DeepLIFT attribution analysis', 'perform in silico evolution'],
    icons = ['check', '', ''],
    layout = widgets.Layout(margin = '0px 0px 20px 0px')
  )

  button_predict = widgets.Button(
    description = 'predict strength',
    layout = widgets.Layout(margin = '0px 0px 20px 0px')
  )

  dl_cb_label = widgets.Label(value = 'DeepLIFT condition:')
  dl_checkboxes = [widgets.Checkbox(value = False, description = out, indent = True) for out in plantGREPcli.model_outputs]
  button_deeplift = widgets.Button(description = 'run DeepLIFT')
  dl_ui = widgets.VBox(
    [dl_cb_label, *dl_checkboxes, button_deeplift],
    layout = widgets.Layout(margin = '0px 0px 20px 0px')
  )

  evo_slider = widgets.IntSlider(
    value = 12,
    min = 1,
    max = 25,
    description = 'Rounds:'
  )
  evo_strong_label = widgets.Label(value = 'Increase strength in:')
  evo_strong_checkboxes = [widgets.Checkbox(value = False, description = out, indent = True) for out in plantGREPcli.model_outputs + ['tobacco']]
  evo_strong = widgets.VBox([evo_strong_label, *evo_strong_checkboxes])
  evo_weak_label = widgets.Label(value = 'Decrease strength in:')
  evo_weak_checkboxes = [widgets.Checkbox(value = False, description = out, indent = True) for out in plantGREPcli.model_outputs + ['tobacco']]
  evo_weak = widgets.VBox([evo_weak_label, *evo_weak_checkboxes])
  evo_objectives = widgets.HBox([evo_strong, evo_weak])
  button_evolution = widgets.Button(description = 'evolve sequence')
  evo_ui = widgets.VBox(
    [evo_slider, evo_objectives, button_evolution],
    layout = widgets.Layout(margin = '0px 0px 20px 0px')
  )

  button_download = widgets.Button(
    description = 'download results',
    layout = widgets.Layout(margin = '0px 0px 50px 0px')
  )
  button_plot_download = widgets.Button(description = 'download plot')

  pred_plot_seq = widgets.Dropdown(
    options = [''],
    value = '',
    description = 'Sequence:'
  )
  pred_plot_cb_label = widgets.Label(value = 'Show conditions:')
  pred_plot_checkboxes = [widgets.Checkbox(value = True, description = out, indent = True) for out in plantGREPcli.model_outputs]
  pred_plot_pos = widgets.IntRangeSlider(
    description = 'Positions:',
    value = [1, 170],
    min = 1,
    max = 170,
    step = 1,
    continuous_update = False,
    layout = widgets.Layout(width = '604px')
  )

  dl_plot_seq = widgets.Dropdown(
    options = [''],
    value = '',
    description = 'Sequence:'
  )
  dl_plot_cond = widgets.Dropdown(
    options = plantGREPcli.model_outputs,
    value = plantGREPcli.model_outputs[0],
    description = 'Condition:'
  )
  dl_plot_pos = widgets.IntRangeSlider(
    description = 'Positions:',
    value = [1, 100],
    min = 1,
    max = 100,
    step = 1,
    continuous_update = False,
    layout = widgets.Layout(width = '604px')
  )

  evo_plot_cb_label = widgets.Label(value = 'Show conditions:')
  evo_plot_checkboxes = [widgets.Checkbox(value = False, description = out, indent = True) for out in plantGREPcli.model_outputs]

  output = widgets.Output()

  # hide deeplift, evolution, and download buttons initially
  button_download.layout.display = 'none'
  dl_ui.layout.display = 'none'
  evo_ui.layout.display = 'none'

  # function to toggle between prediction and DeepLIFT mode
  def toggle_mode(b):
    b.owner.icons = ['check' if b.new == m else '' for m in b.owner.options]
    # output.clear_output()
    with output:
      if b.new == 'prediction':
        # change button/checkbox visibility
        button_predict.layout.display = ''
        dl_ui.layout.display = 'none'
        dl_plot_ui.layout.display = 'none'
        evo_ui.layout.display = 'none'
        evo_plot_ui.layout.display = 'none'
        # display results if there are any
        output.clear_output()
        if len(results[b.new]) > 0:
          display(results[b.new])
          button_download.layout.display = ''
          pred_plot_ui.layout.display = ''
        else:
          button_download.layout.display = 'none'
          pred_plot_ui.layout.display = 'none'
      elif b.new == 'DeepLIFT':
        # change button/checkbox visibility
        button_predict.layout.display = 'none'
        pred_plot_ui.layout.display = 'none'
        dl_ui.layout.display = ''
        evo_ui.layout.display = 'none'
        evo_plot_ui.layout.display = 'none'
        # display results if there are any
        output.clear_output()
        if len(results[b.new]) > 0:
          display(results[b.new])
          button_download.layout.display = ''
          dl_plot_ui.layout.display = ''
        else:
          button_download.layout.display = 'none'
          dl_plot_ui.layout.display = 'none'
      elif b.new == 'evolution':
        # change button/checkbox visibility
        button_predict.layout.display = 'none'
        pred_plot_ui.layout.display = 'none'
        dl_ui.layout.display = 'none'
        dl_plot_ui.layout.display = 'none'
        evo_ui.layout.display = ''
        # display results if there are any
        output.clear_output()
        if len(results[b.new]) > 0:
          display(results[b.new])
          button_download.layout.display = ''
          evo_plot_ui.layout.display = ''
        else:
          button_download.layout.display = 'none'
          evo_plot_ui.layout.display = 'none'

  # function to predict enhancer strength
  def predict(b):
    output.clear_output()
    button_download.layout.display = 'none'
    pred_plot_ui.layout.display = 'none'
    with output:
      if seq_input.value == '':
        raise ValueError('Supply at least one sequence first')
      else:
        # load sequences and predict enhancer strength
        sequence_df = plantGREPcli.read_sequences(string = seq_input.value)
        results['prediction'] = plantGREPcli.predict(model, sequence_df, device)
        display(results['prediction'])
        # show download button
        button_download.layout.display = ''
        # update and show prediction plot ui
        pred_plot_seq.options = results['prediction']['name'].unique().tolist()
        pred_plot_seq.value = pred_plot_seq.options[0]
        pred_plot_update_positions()
        pred_plot_ui.layout.display = ''

  # function to perform DeepLIFT analysis
  def deeplift(b):
    output.clear_output()
    button_download.layout.display = 'none'
    dl_plot_ui.layout.display = 'none'
    conditions = [out for cb, out in zip(dl_checkboxes, plantGREPcli.model_outputs) if cb.value]
    with output:
      if seq_input.value == '':
        raise ValueError('Supply at least one sequence first')
      elif len(conditions) == 0:
        raise ValueError('Select at least one condition')
      else:
        # load sequences and run DeepLIFT
        sequence_df = plantGREPcli.read_sequences(string = seq_input.value)
        results['DeepLIFT'] = plantGREPcli.deeplift(model, sequence_df, device, conditions)
        display(results['DeepLIFT'])
        # show download button
        button_download.layout.display = ''
        # update and show DeepLIFT plot ui
        dl_plot_seq.options = results['DeepLIFT']['name'].unique().tolist()
        dl_plot_seq.value = dl_plot_seq.options[0]
        dl_plot_cond.options = [col[9:] for col in results['DeepLIFT'] if col.startswith('deeplift_')]
        dl_plot_cond.value = dl_plot_cond.options[0]
        dl_plot_update_positions()
        dl_plot_ui.layout.display = ''

  # function for in silico evolution
  def evolution(b):
    output.clear_output()
    button_download.layout.display = 'none'
    evo_plot_ui.layout.display = 'none'
    for cb in evo_plot_checkboxes:
      cb.value = False # reset plot checkboxes to force refresh
    strong = [out for cb, out in zip(evo_strong_checkboxes, plantGREPcli.model_outputs + ['tobacco']) if cb.value]
    weak = [out for cb, out in zip(evo_weak_checkboxes, plantGREPcli.model_outputs + ['tobacco']) if cb.value]
    with output:
      if seq_input.value == '':
        raise ValueError('Supply a 170-bp sequence first')
      elif len(strong) == 0 and len(weak) == 0:
        raise ValueError('Select at least one condition to increase/decrease strength in')
      elif len(set(strong).intersection(weak)) > 0:
        raise ValueError('Each condition can only be selected once')
      else:
        # load sequence and perform evolution
        sequence_df = plantGREPcli.read_sequences(string = seq_input.value)
        results['evolution'] = plantGREPcli.evolve(model, sequence_df, device, evo_slider.value, strong, weak)
        display(results['evolution'])
        # show download button
        button_download.layout.display = ''
        # update and show evolution plot ui
        for cb, out in zip(evo_plot_checkboxes, plantGREPcli.model_outputs):
          cb.value = (out in strong + weak) or ('tobacco' in strong + weak and out in plantGREPcli.model_outputs[:4])
        evo_plot_ui.layout.display = ''

  # function to download results
  def result_download(b):
    results[tb_mode.value].to_csv(f'{tb_mode.value}_results.tsv', sep = '\t', index = False)
    files.download(f'{tb_mode.value}_results.tsv')

  # function to plot enhancer strength predictions
  def plot_prediction(sequence, positions, **kwargs):
    filtered_data = results['prediction'][results['prediction']['name'] == sequence]
    filtered_data['position'] = (filtered_data['start'] + filtered_data['end']) * 0.5
    filtered_data = filtered_data[filtered_data['position'].between(*positions)]
    conditions = [c for c, b in kwargs.items() if b]
    pred_plot = filtered_data.plot(
      kind = 'bar' if len(filtered_data) == 1 else 'line',
      x = 'name' if len(filtered_data) == 1 else 'position',
      y = [f'prediction_{c}' for c in conditions],
      figsize = (10, 5)
    )
    pred_plot.legend(conditions)
    pred_plot.axhline(color = 'black', linewidth = 0.5)
    pred_plot.set_ylabel('predicted log$_2$(enhancer strength)')
    pred_plot.tick_params(axis = 'x', labelrotation = 0)
    seqname = ''.join(x for x in sequence if (x.isalnum() or x in '._- ()'))
    plots['prediction']['plot'] = pred_plot.get_figure()
    plots['prediction']['filename'] = f'prediction_{seqname}_{positions[0]}-{positions[1]}.pdf'

  # function to plot DeepLIFT results
  def plot_deeplift(sequence, condition, positions):
    filtered_data = results['DeepLIFT'][(results['DeepLIFT']['name'] == sequence) & (results['DeepLIFT']['position'].between(*positions))]
    deeplift_matrix = filtered_data.pivot(index = 'position', columns = 'base', values = f'deeplift_{condition}').fillna(0)
    dl_logo = logomaker.Logo(deeplift_matrix)
    dl_logo.ax.set_ylim(results['DeepLIFT'][results['DeepLIFT']['name'] == sequence][f'deeplift_{condition}'].agg(['min', 'max']).tolist())
    dl_logo.ax.set_xlabel('position')
    dl_logo.ax.set_ylabel(f'contribution score ({condition})')
    seqname = ''.join(x for x in sequence if (x.isalnum() or x in '._- ()'))
    plots['DeepLIFT']['plot'] = dl_logo.fig
    plots['DeepLIFT']['filename'] = f'DeepLIFT_{condition}_{seqname}_{positions[0]}-{positions[1]}.pdf'

  # function to plot evolution results
  def plot_evolution(**kwargs):
    filtered_data = results['evolution']
    conditions = [c for c, b in kwargs.items() if b]
    evo_plot = filtered_data.plot(
      kind = 'line',
      x = 'round',
      y = [f'prediction_{c}' for c in conditions],
      figsize = (10, 5)
    )
    evo_plot.legend(conditions)
    evo_plot.axhline(color = 'black', linewidth = 0.5)
    evo_plot.set_xlabel('in silico evolution round')
    evo_plot.set_ylabel('predicted log$_2$(enhancer strength)')
    evo_plot.tick_params(axis = 'x', labelrotation = 0)
    plots['evolution']['plot'] = evo_plot.get_figure()
    plots['evolution']['filename'] = 'evolution.pdf'


  # function to update position slider for prediction plots
  def pred_plot_update_positions(*args):
    pred_plot_pos.min = results['prediction'][results['prediction']['name'] == pred_plot_seq.value]['start'].min()
    pred_plot_pos.max = results['prediction'][results['prediction']['name'] == pred_plot_seq.value]['end'].max()
    pred_plot_pos.value = [pred_plot_pos.min, pred_plot_pos.max]

  # function to update position slider for DeepLIFT plots
  def dl_plot_update_positions(*args):
    dl_plot_pos.min = results['DeepLIFT'][results['DeepLIFT']['name'] == dl_plot_seq.value]['position'].min()
    dl_plot_pos.max = results['DeepLIFT'][results['DeepLIFT']['name'] == dl_plot_seq.value]['position'].max()

  # function to download plot
  def plot_download(b):
    plots[tb_mode.value]['plot'].savefig(plots[tb_mode.value]['filename'], bbox_inches = 'tight')
    files.download(plots[tb_mode.value]['filename'])

  # assign functions to buttons
  tb_mode.observe(toggle_mode, 'value')
  button_predict.on_click(predict)
  button_deeplift.on_click(deeplift)
  button_evolution.on_click(evolution)
  button_download.on_click(result_download)
  button_plot_download.on_click(plot_download)
  pred_plot_seq.observe(pred_plot_update_positions, 'value')
  dl_plot_seq.observe(dl_plot_update_positions, 'value')

  # set up prediction plot widgets
  pred_plot = widgets.interactive_output(plot_prediction, {'sequence' : pred_plot_seq, 'positions' : pred_plot_pos, **dict(zip(plantGREPcli.model_outputs, pred_plot_checkboxes))})
  pred_plot_ui = widgets.VBox([pred_plot_seq, pred_plot_pos, pred_plot_cb_label, *pred_plot_checkboxes, pred_plot, button_plot_download])
  pred_plot_ui.layout.display = 'none'# hide initially

  # set up DeepLIFT plot widgets
  dl_plot = widgets.interactive_output(plot_deeplift, {'sequence' : dl_plot_seq, 'condition' : dl_plot_cond, 'positions' : dl_plot_pos})
  dl_plot_dds = widgets.HBox([dl_plot_seq, dl_plot_cond])
  dl_plot_ui = widgets.VBox([dl_plot_dds, dl_plot_pos, dl_plot, button_plot_download])
  dl_plot_ui.layout.display = 'none'# hide initially

  # set up evolution plot widgets
  evo_plot = widgets.interactive_output(plot_evolution, {**dict(zip(plantGREPcli.model_outputs, evo_plot_checkboxes))})
  evo_plot_ui = widgets.VBox([evo_plot_cb_label, *evo_plot_checkboxes, evo_plot, button_plot_download])
  evo_plot_ui.layout.display = 'none'# hide initially

  # make Font Awesome available
  display(HTML('<link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css">'))

  # display widgets
  display(seq_input, tb_mode, button_predict, dl_ui, evo_ui, output, button_download, pred_plot_ui, dl_plot_ui, evo_plot_ui)

except ModuleNotFoundError:
  # install missing modules and kill session (restart is required because of a change in the numpy version)
  import os
  import time
  import ipywidgets as widgets
  from IPython.display import display, HTML
  output = widgets.Output()
  display(output)
  with output:
    display(HTML('Installing new modules. This takes about 2-3 minutes. Please be patient :)'))
  !uv pip install --system pytorch_lightning seqpro captum itables logomaker
  output.clear_output()
  with output:
    display(HTML('<span style="font-size:150%">New modules were installed. <span style="color: #ff0000"><b>Please re-run this cell!</b></span></span>'))
  time.sleep(0.001)
  os.kill(os.getpid(), 9)

## FAQs

1. What are the sequence requirements?

    Sequences must be supplied in [FASTA format](https://en.wikipedia.org/wiki/FASTA_format), contain only A, C, G, and T nucleotides, and must be at least 170-bp long (shorter sequnces will be ignored). For sequences longer than 170 bp, you will get an enhancer strength prediction for every possible 170-bp fragment of it.
    
    For the `evolution` function, only a single, 170-bp sequence can be used.

1. What do the results mean?

    The `prediction` and `evolution` functions return the predicted enhancer strength (log<sub>2</sub>-transformed and normalized to a no-enhancer control) in the indicated condition.

    The `DeepLIFT` function returns the contribution score for each base indicating how much this base contributes to the overall enhancer strength prediction. Positive values are associated with increased enhancer strength, negative values with decreased strength.

1. What does `light`, `dark`, `warm`, `cold`, and `maize` mean?

    In our experiments, we measured the enhancer strength of candidate sequences in different assay systems and under different environmental conditions. Experiments were conducted with tobacco plants kept in normal light/dark cycles (`light`), in complete darkness (`dark`), and at elevated (`warm`) or reduced (`cold`) ambient temperature. Additionally, we also conducted experiments in maize protoplasts (`maize`). The plantGREP model was trained to predict enhancer strength in all these conditions/assay systems.
    
    Please read [Jores et al., 2025, unpublished](<DOI>) for more information.

1. What is *in silico* evolution?

    *In silico* evolution is a method to generate strong enhancers. For this, all possible single-nucleotide substitution variants of a given sequence are scored with the plantGREP model. The highest-scoring sequence variant is then used as the starting point for the next round, with each iteration introducing a single mutation that should improve enhancer strength.

1. I accidentally clicked `Show code`. How can I hide the code again?

    Right-click on any line of code and select `Hide code`

1. Running the model takes a long time. What is happening?

    Initializing the notebook takes relatively long, and there is nothing much one can do about it.
    
    If the `prediction`, `DeepLIFT`, or `evolution` function takes long, you might be connected to a runtime without GPU access. To enable GPU access, click on `Runtime` > `Change runtime type`, under `Hardware accelerator` select `T4 GPU` , and click `save`.